# Estudo de Caso 9.X2 — Caldeira Aquatubular: Dimensionamento Térmico com Bell-Delaware

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 9 — Trocadores de Calor |
| **Nível** | Graduação/Pós-Graduação |
| **Tempo estimado** | 90 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | LMTD em duas etapas + ε-NTU + Bell-Delaware |
| **Bibliografia** | Incropera et al. (2017), Cap. 11; Kakaç, Liu & Majumdar (2012), Cap. 3–4; Shah & Sekulić (2003), Cap. 2 |

---

## 📋 Resumo

Dimensionamos uma caldeira **aquatubular** para uma pequena indústria alimentícia, mantendo as mesmas dimensões e capacidade da caldeira flamotubular do Estudo de Caso 9.7. Na configuração aquatubular, a água escoa **dentro dos tubos** (escoamento interno forçado com ebulição) e os gases quentes passam **pelo casco** em escoamento cruzado com chicanas segmentais. Aplicamos o método de Bell-Delaware para o lado dos gases e Dittus-Boelter/Chen para o lado da água, com validação cruzada pelos métodos LMTD e ε-NTU.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Identificar** corretamente a configuração física aquatubular e suas implicações nas correlações convectivas
2. **Aplicar** o método de Bell-Delaware para escoamento cruzado com chicanas no casco
3. **Calcular** coeficientes convectivos internos (Dittus-Boelter para aquecimento da água)
4. **Determinar** o coeficiente global U considerando resistências em série e incrustação
5. **Aplicar** o método LMTD em duas etapas (pré-aquecimento + vaporização)
6. **Validar** o dimensionamento pelo método ε-NTU
7. **Comparar** o desempenho térmico com a configuração flamotubular equivalente

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## 🏭 Seção 1: Parâmetros da Caldeira Aquatubular

### Contexto Industrial

Uma caldeira **aquatubular** é um trocador de calor onde:
- A **água escoa DENTRO dos tubos** (escoamento interno forçado)
- Os **gases quentes passam PELO CASCO** em escoamento cruzado com chicanas

Esta configuração é típica de caldeiras de **médio e grande porte**, pois permite pressões de operação mais elevadas (os tubos de pequeno diâmetro resistem melhor à pressão interna que um casco de grande diâmetro).

> ⚠️ **Ponto crítico:** A escolha da configuração define as correlações convectivas adequadas. Em caldeiras aquatubulares, os gases escoam **transversalmente** ao feixe de tubos, com chicanas segmentais. Portanto, o método de **Bell-Delaware** (ou Zukauskas) é o adequado para o lado dos gases, enquanto **Dittus-Boelter** é usado para a água dentro dos tubos.

> 💡 **Observação:** Este estudo de caso é o **complemento** do EC 9.7 (Caldeira Flamotubular). Mantemos as mesmas dimensões e capacidade para permitir comparação direta entre as configurações.

### Normas de Segurança

No Brasil, o projeto e operação de caldeiras deve obedecer à **NR13** do Ministério do Trabalho. Requisitos essenciais:
1. Válvula de segurança (evitar sobrepressão)
2. Manômetro (indicação da pressão do vapor)
3. Sistema de alimentação de água
4. Sistema de drenagem rápida
5. Controle automático de nível

### Tabela de Parâmetros (mesmos do EC 9.7)

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| **Fluido quente (gases no casco)** | | | |
| Vazão mássica | $\dot{m}_h$ | 1,2 | kg/s |
| Temperatura de entrada | $T_{h,in}$ | 900 | °C |
| Temperatura de saída (estimada) | $T_{h,out}$ | 250 | °C |
| Calor específico | $c_{p,h}$ | 1100 | J/(kg·K) |
| Viscosidade dinâmica | $\mu_h$ | 4,5×10⁻⁵ | Pa·s |
| Condutividade térmica | $k_h$ | 0,065 | W/(m·K) |
| Número de Prandtl | $Pr_h$ | 0,72 | – |
| Densidade média (~600°C) | $\rho_h$ | 0,35 | kg/m³ |
| **Fluido frio (água nos tubos)** | | | |
| Pressão de operação | $p$ | 10 | bar |
| Temperatura de saturação | $T_{sat}$ | 179,9 | °C |
| Vazão de vapor desejada | $\dot{m}_v$ | 0,3 | kg/s |
| Calor latente de vaporização | $h_{fg}$ | 2015 | kJ/kg |
| Temperatura da água de alimentação | $T_{c,in}$ | 80 | °C |
| Calor específico da água | $c_{p,w}$ | 4180 | J/(kg·K) |
| Viscosidade dinâmica da água | $\mu_w$ | 3,5×10⁻⁴ | Pa·s |
| Condutividade térmica da água | $k_w$ | 0,67 | W/(m·K) |
| **Geometria (igual ao EC 9.7)** | | | |
| Diâmetro interno dos tubos | $D_i$ | 25 | mm |
| Diâmetro externo dos tubos | $D_o$ | 30 | mm |
| Comprimento dos tubos | $L$ | 3,0 | m |
| Número de tubos | $N_t$ | 180 | – |
| Condutividade do aço carbono | $k_{tubo}$ | 45 | W/(m·K) |
| **Casco e chicanas (NOVO)** | | | |
| Diâmetro do casco | $D_s$ | 0,600 | m |
| Espaçamento entre chicanas | $L_b$ | 0,300 | m |
| Passo dos tubos (triangular) | $p_T$ | 40 | mm |
| Corte da chicana | $BC$ | 25 | % |
| **Incrustação** | | | |
| Fator interno (água) | $R_{f,i}$ | 0,0002 | m²·K/W |
| Fator externo (gases) | $R_{f,o}$ | 0,0004 | m²·K/W |
| **Rendimento** | | | |
| Rendimento da combustão | $\eta_{comb}$ | 0,92 | – |

In [ ]:
# ============================================================
# PARÂMETROS DA CALDEIRA AQUATUBULAR
# ============================================================

# --- Fluido Quente (Gases de Combustão) - AGORA NO CASCO ---
m_h = 1.2              # Vazão mássica dos gases [kg/s]
T_h_in = 900.0         # Temperatura de entrada dos gases [°C]
T_h_out_est = 250.0    # Temperatura de saída estimada [°C]
cp_h = 1100.0          # Calor específico dos gases [J/(kg·K)]
mu_h = 4.5e-5          # Viscosidade dinâmica dos gases [Pa·s]
k_h = 0.065            # Condutividade térmica dos gases [W/(m·K)]
Pr_h = 0.72            # Número de Prandtl dos gases [-]
rho_h = 0.35           # Densidade média dos gases [kg/m³]
eta_comb = 0.92        # Rendimento da combustão [-]

# --- Fluido Frio (Água/Vapor) - AGORA DENTRO DOS TUBOS ---
P_op = 10.0            # Pressão de operação [bar]
T_sat = 179.9          # Temperatura de saturação a 10 bar [°C]
m_v = 0.3              # Vazão de vapor desejada [kg/s]
h_fg = 2015e3          # Calor latente de vaporização [J/kg]
T_c_in = 80.0          # Temperatura da água de alimentação [°C]
cp_w = 4180.0          # Calor específico da água [J/(kg·K)]
mu_w = 3.5e-4          # Viscosidade dinâmica da água [Pa·s]
k_w = 0.67             # Condutividade térmica da água [W/(m·K)]
rho_w = 887.0          # Densidade da água a ~180°C [kg/m³]
Pr_w = cp_w * mu_w / k_w  # Número de Prandtl da água [-]

# --- Geometria dos Tubos (IGUAL ao EC 9.7) ---
D_i = 0.025            # Diâmetro interno dos tubos [m]
D_o = 0.030            # Diâmetro externo dos tubos [m]
k_tubo = 45.0          # Condutividade do aço carbono [W/(m·K)]
N_t = 180              # Número de tubos [-]
L_tubo = 3.0           # Comprimento dos tubos [m]

# --- Geometria do Casco e Chicanas (NOVO - aquatubular) ---
D_s = 0.600            # Diâmetro do casco [m]
L_b = 0.300            # Espaçamento entre chicanas [m]
p_T = 0.040            # Passo dos tubos (arranjo triangular) [m]
baffle_cut = 0.25      # Corte da chicana (25%) [-]

# --- Fatores de Incrustação ---
R_fi = 0.0002          # Fator interno (água nos tubos) [m²·K/W]
R_fo = 0.0004          # Fator externo (gases no casco) [m²·K/W]

# --- Áreas de Transferência ---
A_i = np.pi * D_i * L_tubo * N_t   # Área interna total (água) [m²]
A_o = np.pi * D_o * L_tubo * N_t   # Área externa total (gases) [m²]

print('=' * 60)
print('DADOS DE ENTRADA — CALDEIRA AQUATUBULAR')
print('=' * 60)
print(f'\n🔥 Configuração Física:')
print(f'  • Água/Vapor: DENTRO dos tubos (Escoamento Interno)')
print(f'  • Gases quentes: NO CASCO (Escoamento Cruzado com Chicanas)')
print(f'\n💧 Dados da Água/Vapor (tubos):')
print(f'  • Vazão de vapor: {m_v} kg/s')
print(f'  • Pressão: {P_op} bar | T_sat: {T_sat}°C')
print(f'  • T_água_alim: {T_c_in}°C')
print(f'  • Pr_água: {Pr_w:.2f}')
print(f'\n🔥 Dados dos Gases (casco):')
print(f'  • Vazão mássica: {m_h} kg/s')
print(f'  • T_entrada: {T_h_in}°C | T_saída_est: {T_h_out_est}°C')
print(f'  • c_p: {cp_h} J/(kg·K) | ρ: {rho_h} kg/m³')
print(f'\n📐 Geometria:')
print(f'  • Nº tubos: {N_t} | Comprimento: {L_tubo} m')
print(f'  • D_i = {D_i*1000:.0f} mm | D_o = {D_o*1000:.0f} mm')
print(f'  • Diâmetro do casco: D_s = {D_s*1000:.0f} mm')
print(f'  • Espaçamento chicanas: L_b = {L_b*1000:.0f} mm')
print(f'  • Passo dos tubos: p_T = {p_T*1000:.0f} mm (triangular)')
print(f'  • Área interna total (A_i): {A_i:.2f} m²')
print(f'  • Área externa total (A_o): {A_o:.2f} m²')

---
## 🔬 Seção 2: Verificação — Balanço Energético

Antes do dimensionamento, verificamos o balanço de energia da caldeira.

### Carga térmica requerida

$$\dot{Q}_{vap} = \dot{m}_v \cdot h_{fg}$$

$$\dot{Q}_{pre} = \dot{m}_v \cdot c_{p,w} \cdot (T_{sat} - T_{c,in})$$

$$\dot{Q}_{req} = \dot{Q}_{vap} + \dot{Q}_{pre}$$

### Calor disponível dos gases

$$\dot{Q}_{disp} = \eta_{comb} \cdot \dot{m}_h \cdot c_{p,h} \cdot (T_{h,in} - T_{h,out})$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 11, Seção 11.2.

In [ ]:
# ============================================================
# BALANÇO ENERGÉTICO
# ============================================================

t_inicio = time.time()

# Carga térmica de vaporização
Q_vap = m_v * h_fg                       # [W]

# Carga térmica de pré-aquecimento
Q_pre = m_v * cp_w * (T_sat - T_c_in)    # [W]

# Carga térmica total requerida
Q_req = Q_vap + Q_pre                    # [W]

# Calor disponível dos gases
Q_disp = eta_comb * m_h * cp_h * (T_h_in - T_h_out_est)   # [W]

# Margem térmica
margem_termica = (Q_disp / Q_req - 1.0) * 100.0            # [%]

# Temperatura de saída real dos gases (se toda carga for transferida)
T_h_out_real = T_h_in - Q_req / (eta_comb * m_h * cp_h)

t_cpu_balanco = time.time() - t_inicio

print('=' * 60)
print('BALANÇO ENERGÉTICO — CALDEIRA AQUATUBULAR')
print('=' * 60)
print(f'\n📊 Cargas térmicas:')
print(f'  • Pré-aquecimento (Q_pre):  {Q_pre/1e3:.2f} kW ({Q_pre/Q_req*100:.1f}%)')
print(f'  • Vaporização (Q_vap):      {Q_vap/1e3:.2f} kW ({Q_vap/Q_req*100:.1f}%)')
print(f'  • Total requerido (Q_req):  {Q_req/1e3:.2f} kW')
print(f'\n🔥 Calor disponível dos gases:')
print(f'  • Q_disp = {Q_disp/1e3:.2f} kW')
print(f'\n🌡️ Temperatura de saída real dos gases:')
print(f'  • T_h,out = {T_h_out_real:.1f}°C (vs. estimativa de {T_h_out_est}°C)')
print(f'\n📏 Margem térmica: {margem_termica:.1f}%')

assert Q_disp >= Q_req, '❌ Calor disponível insuficiente!'
print('\n✅ Balanço energético verificado — calor disponível é suficiente')
print(f'⏱️ Tempo de CPU: {t_cpu_balanco:.6f} s')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Lado da Água (Dentro dos Tubos) — Dittus-Boelter para Aquecimento

Para escoamento turbulento interno com aquecimento:

$$Nu_D = 0{,}023 \cdot Re_D^{0{,}8} \cdot Pr^{0{,}4}$$

onde:
- $Re_D = \frac{\rho_w u D_i}{\mu_w}$
- $u = \frac{\dot{m}_v}{\rho_w A_{flow}}$
- $A_{flow} = N_t \frac{\pi D_i^2}{4}$

### 3.2 Lado dos Gases (Casco com Chicanas) — Bell-Delaware

O método de Bell-Delaware calcula o coeficiente convectivo por:

$$h_o = h_{ideal} \cdot J_c \cdot J_l \cdot J_b \cdot J_s \cdot J_w$$

onde:
- $h_{ideal}$: coeficiente para escoamento cruzado ideal (Zukauskas)
- $J_c$: fator de correção para corte da chicana
- $J_l$: fator de correção para vazamento chicana-casco
- $J_b$: fator de correção para bypass feixe-casco
- $J_s$: fator de correção para espaçamento não-uniforme das chicanas
- $J_w$: fator de correção para escoamento na janela da chicana

### 3.3 Correlação de Zukauskas para Escoamento Cruzado

$$Nu_D = C \cdot Re_D^m \cdot Pr^{0{,}36} \cdot \left(\frac{Pr}{Pr_s}\right)^{0{,}25}$$

Para arranjo triangular e $10^3 < Re_D < 2 \times 10^5$: $C = 0{,}35$, $m = 0{,}6$.

### 3.4 Coeficiente Global U (base externa)

$$\frac{1}{U_o A_o} = \frac{1}{h_i A_i} + \frac{R_{f,i}}{A_i} + \frac{\ln(D_o/D_i)}{2\pi k_{tubo} L N_t} + \frac{R_{f,o}}{A_o} + \frac{1}{h_o A_o}$$

### 3.5 LMTD em Duas Etapas

Como a água passa por duas fases distintas (pré-aquecimento + vaporização), a LMTD é calculada separadamente para cada zona, com fator de correção $F$ para escoamento cruzado.

> 📖 **Referência:** Incropera et al. (2017), Cap. 11; Kakaç, Liu & Majumdar (2012), Cap. 4; Shah & Sekulić (2003), Cap. 2.

---
## ⚡ Seção 4: Critério Numérico e Validação das Correlações

> 💡 **Observação:** Neste problema de regime permanente com solução algébrica, não se aplica a condição CFL. Os critérios de adequação são:

1. **Dittus-Boelter:** válido para $Re_D > 10^4$, $0{,}6 < Pr < 160$, $L/D > 10$.

2. **Zukauskas:** válido para $10^3 < Re_D < 2 \times 10^5$ em escoamento cruzado.

3. **Velocidade dos gases:** Em caldeiras aquatubulares, a velocidade típica dos gases no casco é 10–30 m/s. Velocidades muito altas causam erosão dos tubos.

4. **Velocidade da água:** Para evitar erosão e vibrações, $u_{água} < 3$ m/s em tubos de aço carbono.

5. **Configuração física:** Bell-Delaware é **fisicamente adequado** para caldeiras aquatubulares (gases no casco com chicanas).

In [ ]:
# ============================================================
# VERIFICAÇÃO — CRITÉRIOS DE VALIDADE
# ============================================================

# Área de escoamento da água nos tubos
A_flow_agua = N_t * np.pi * D_i**2 / 4.0   # [m²]

# Velocidade da água nos tubos
u_agua = m_v / (rho_w * A_flow_agua)   # [m/s]

# Reynolds da água
Re_agua = rho_w * u_agua * D_i / mu_w   # [-]

# Área de escoamento dos gases no casco (Bell-Delaware)
S_m = (p_T - D_o) * L_b * D_s / p_T   # [m²]

# Velocidade dos gases no casco
u_gases = m_h / (rho_h * S_m)   # [m/s]

# Reynolds dos gases
Re_gases = rho_h * u_gases * D_o / mu_h   # [-]

print('⚡ VERIFICAÇÃO DOS CRITÉRIOS DE VALIDADE')
print('=' * 60)
print(f'\n💧 Lado da Água (tubos):')
print(f'  • Área de escoamento: {A_flow_agua:.4f} m²')
print(f'  • Velocidade: u = {u_agua:.2f} m/s (recomendado < 3 m/s)')
print(f'  • Reynolds: Re = {Re_agua:.0f}')

if Re_agua >= 10000:
    print(f'  ✅ Dittus-Boelter VÁLIDO (Re > 10⁴)')
elif Re_agua >= 4000:
    print(f'  ⚠️ Regime de transição — usar Gnielinski')
else:
    print(f'  ❌ Regime laminar — Dittus-Boelter NÃO aplicável')

if u_agua > 3.0:
    print(f'  ⚠️ Velocidade da água elevada (> 3 m/s) — risco de erosão')
else:
    print(f'  ✅ Velocidade da água adequada')

print(f'\n🔥 Lado dos Gases (casco com chicanas):')
print(f'  • Área de escoamento cruzado: S_m = {S_m:.4f} m²')
print(f'  • Velocidade: u = {u_gases:.2f} m/s (recomendado 10-30 m/s)')
print(f'  • Reynolds: Re = {Re_gases:.0f}')

if 1000 <= Re_gases <= 2e5:
    print(f'  ✅ Zukauskas/Bell-Delaware VÁLIDO')
else:
    print(f'  ⚠️ Reynolds fora da faixa de validade')

if u_gases > 40.0:
    print(f'  ⚠️ Velocidade dos gases muito alta — risco de erosão')
else:
    print(f'  ✅ Velocidade dos gases adequada')

---
## 🌪️ Seção 5: Condição de Entrada (Forçamento Térmico)

Neste problema de regime permanente, o forçamento é a **temperatura de entrada dos gases**:

$$T_{h,in} = 900°C$$

e a **demanda de vapor** no lado da água:

$$\dot{m}_v = 0{,}3 \text{ kg/s a } p = 10 \text{ bar}$$

> 💡 **Observação:** A temperatura de saturação $T_{sat} = 179{,}9°C$ é fixada pela pressão de operação. Durante a vaporização dentro dos tubos (flow boiling), a temperatura da água permanece aproximadamente constante.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# PASSO 1: COEFICIENTE CONVECTIVO INTERNO (ÁGUA NOS TUBOS)
# Dittus-Boelter para AQUECIMENTO (n = 0,4)
# ============================================================

# Nusselt por Dittus-Boelter (aquecimento: n = 0.4)
Nu_agua = 0.023 * Re_agua**0.8 * Pr_w**0.4

# Coeficiente convectivo interno
h_i_agua = Nu_agua * k_w / D_i   # [W/(m²·K)]

print('=' * 60)
print('PASSO 1: COEFICIENTE INTERNO — ÁGUA NOS TUBOS')
print('=' * 60)
print(f'\n💧 Lado dos Tubos (Água) — ESCOAMENTO INTERNO:')
print(f'  • Área de escoamento (A_flow): {A_flow_agua:.4f} m²')
print(f'  • Velocidade da água (u): {u_agua:.2f} m/s')
print(f'  • Reynolds (Re_D): {Re_agua:.0f}')

if Re_agua >= 10000:
    print(f'  • Regime: Turbulento (Re > 10.000)')
    print(f'  • Correlação: Dittus-Boelter (aquecimento, n=0,4)')
else:
    print(f'  • Regime: Transição (Re < 10.000)')
    print(f'  • Nota: Gnielinski seria mais adequado.')

print(f'  • Nusselt (Nu_D): {Nu_agua:.1f}')
print(f'  • Coeficiente: h_i = {h_i_agua:.1f} W/(m²·K)')

In [ ]:
# ============================================================
# PASSO 2: COEFICIENTE CONVECTIVO EXTERNO (GASES NO CASCO)
# Método de Bell-Delaware
# ============================================================

# --- 2.1 Coeficiente ideal por Zukauskas (escoamento cruzado) ---
# Para arranjo triangular e 10^3 < Re < 2×10^5:
# C = 0,35 e m = 0,6
C_zuk = 0.35
m_zuk = 0.6

# Prandtl na superfície (aproximação: usar Pr dos gases)
Pr_s = Pr_h

# Nusselt ideal
Nu_ideal = C_zuk * Re_gases**m_zuk * Pr_h**0.36 * (Pr_h / Pr_s)**0.25

# Coeficiente ideal
h_ideal = Nu_ideal * k_h / D_o   # [W/(m²·K)]

# --- 2.2 Fatores de correção de Bell-Delaware ---

# Fator J_c (corte da chicana)
# Para corte de 25%: J_c ≈ 0,55 + 0,72·F_c, onde F_c = 1 - 2·BC
F_c = 1.0 - 2.0 * baffle_cut
J_c = 0.55 + 0.72 * F_c

# Fator J_l (vazamento chicana-casco)
# Valor típico: 0,7 a 1,0. Adotamos 0,9 (folga pequena)
J_l = 0.90

# Fator J_b (bypass feixe-casco)
# Valor típico: 0,8 a 1,0. Adotamos 0,95 (selos laterais)
J_b = 0.95

# Fator J_s (espaçamento não-uniforme das chicanas)
# Para espaçamento uniforme: J_s = 1,0
J_s = 1.0

# Fator J_w (escoamento na janela da chicana)
# Para escoamento turbulento bem distribuído: J_w ≈ 0,95
J_w = 0.95

# Produto dos fatores de correção
J_total = J_c * J_l * J_b * J_s * J_w

# --- 2.3 Coeficiente convectivo externo corrigido ---
h_o_gases = h_ideal * J_total   # [W/(m²·K)]

print('=' * 60)
print('PASSO 2: COEFICIENTE EXTERNO — GASES NO CASCO')
print('=' * 60)
print(f'\n🔥 Lado do Casco (Gases) — ESCOAMENTO CRUZADO:')
print(f'  • Área de escoamento cruzado: S_m = {S_m:.4f} m²')
print(f'  • Velocidade dos gases (u): {u_gases:.2f} m/s')
print(f'  • Reynolds (Re_D): {Re_gases:.0f}')
print(f'  • Correlação: Zukauskas (C={C_zuk}, m={m_zuk})')
print(f'  • Nusselt ideal: {Nu_ideal:.1f}')
print(f'  • h_ideal = {h_ideal:.1f} W/(m²·K)')
print(f'\n📊 Fatores de Bell-Delaware:')
print(f'  • J_c (corte da chicana): {J_c:.4f}')
print(f'  • J_l (vazamento): {J_l:.4f}')
print(f'  • J_b (bypass): {J_b:.4f}')
print(f'  • J_s (espaçamento): {J_s:.4f}')
print(f'  • J_w (janela): {J_w:.4f}')
print(f'  • Produto total: {J_total:.4f}')
print(f'\n  • Coeficiente corrigido: h_o = {h_o_gases:.1f} W/(m²·K)')

In [ ]:
# ============================================================
# PASSO 3: COEFICIENTE GLOBAL U
# ============================================================

# Resistências térmicas
R_conv_i = 1.0 / (h_i_agua * A_i)                              # Convecção interna (água)
R_fouling_i = R_fi / A_i                                       # Incrustação interna
R_wall = np.log(D_o / D_i) / (2.0 * np.pi * k_tubo * L_tubo * N_t)  # Parede do tubo
R_fouling_o = R_fo / A_o                                       # Incrustação externa (gases)
R_conv_o = 1.0 / (h_o_gases * A_o)                             # Convecção externa (gases)

# Resistência total
R_total = R_conv_i + R_fouling_i + R_wall + R_fouling_o + R_conv_o

# Coeficiente global (base externa)
U_o = 1.0 / (R_total * A_o)   # [W/(m²·K)]

print('=' * 60)
print('PASSO 3: COEFICIENTE GLOBAL U')
print('=' * 60)
print(f'\n📊 Áreas:')
print(f'  • Área interna (água): A_i = {A_i:.2f} m²')
print(f'  • Área externa (gases): A_o = {A_o:.2f} m²')
print(f'\n🧮 Resistências térmicas [K/W]:')
print(f'  • R_conv,i (água): {R_conv_i:.6f} ({R_conv_i/R_total*100:.1f}%)')
print(f'  • R_fouling,i: {R_fouling_i:.6f} ({R_fouling_i/R_total*100:.1f}%)')
print(f'  • R_wall (tubo): {R_wall:.6f} ({R_wall/R_total*100:.1f}%)')
print(f'  • R_fouling,o: {R_fouling_o:.6f} ({R_fouling_o/R_total*100:.1f}%)')
print(f'  • R_conv,o (gases): {R_conv_o:.6f} ({R_conv_o/R_total*100:.1f}%)')
print(f'  • R_total: {R_total:.6f}')
print(f'\n📏 Coeficiente global:')
print(f'  • U_o = {U_o:.1f} W/(m²·K)')

---
## 🌡️ Seção 7: Visualização — LMTD em Duas Etapas

In [ ]:
# ============================================================
# PASSO 4: LMTD EM DUAS ETAPAS
# ============================================================

# Temperatura intermediária dos gases (fim do pré-aquecimento)
T_h_int = T_h_in - Q_pre / (eta_comb * m_h * cp_h)   # [°C]

# --- ZONA 1: PRÉ-AQUECIMENTO ---
# Água: T_c_in (80°C) → T_sat (179,9°C)
# Gases: T_h_in (900°C) → T_h_int
# Configuração: contracorrente
dT1_pre = T_h_in - T_sat          # Extremidade quente
dT2_pre = T_h_int - T_c_in        # Extremidade fria

if np.isclose(dT1_pre, dT2_pre):
    LMTD_pre = dT1_pre
else:
    LMTD_pre = (dT1_pre - dT2_pre) / np.log(dT1_pre / dT2_pre)

# --- ZONA 2: VAPORIZAÇÃO ---
# Água: T_sat constante (mudança de fase)
# Gases: T_h_int → T_h_out_real
dT1_vap = T_h_int - T_sat         # Extremidade quente
dT2_vap = T_h_out_real - T_sat    # Extremidade fria

if np.isclose(dT1_vap, dT2_vap):
    LMTD_vap = dT1_vap
else:
    LMTD_vap = (dT1_vap - dT2_vap) / np.log(dT1_vap / dT2_vap)

# Fator de correção F para escoamento cruzado (aproximação)
# Para caldeiras aquatubulares com múltiplas passadas: F ≈ 0,9
F_correcao = 0.90

print('=' * 60)
print('PASSO 4: LMTD EM DUAS ETAPAS')
print('=' * 60)
print(f'\n🌡️ Temperatura intermediária dos gases:')
print(f'  • T_h_int = {T_h_int:.2f}°C')
print(f'\n🔹 ZONA 1: PRÉ-AQUECIMENTO')
print(f'  • ΔT₁ = {dT1_pre:.2f}°C')
print(f'  • ΔT₂ = {dT2_pre:.2f}°C')
print(f'  • LMTD_pré = {LMTD_pre:.2f}°C')
print(f'\n🔹 ZONA 2: VAPORIZAÇÃO')
print(f'  • ΔT₁ = {dT1_vap:.2f}°C')
print(f'  • ΔT₂ = {dT2_vap:.2f}°C')
print(f'  • LMTD_vap = {LMTD_vap:.2f}°C')
print(f'\n📊 Fator de correção (escoamento cruzado): F = {F_correcao}')

In [ ]:
# ============================================================
# GRÁFICO: PERFIL DE TEMPERATURA (DUAS ETAPAS)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Eixo x: posição no trocador (0 a 1)
x = np.linspace(0, 1, 100)

# Fração do trocador ocupada pelo pré-aquecimento
x_pre = Q_pre / Q_req

# Perfil dos gases (resfriamento)
T_gases = np.where(x < x_pre,
                   T_h_in - (T_h_in - T_h_int) * (x / x_pre),
                   T_h_int - (T_h_int - T_h_out_real) * ((x - x_pre) / (1 - x_pre)))

# Perfil da água (pré-aquecimento + vaporização)
T_agua = np.where(x < x_pre,
                  T_c_in + (T_sat - T_c_in) * (x / x_pre),
                  T_sat)  # Temperatura constante na vaporização

ax.plot(x, T_gases, 'r-', linewidth=3, label='Gases (casco)')
ax.plot(x, T_agua, 'b-', linewidth=3, label='Água/Vapor (tubos)')

# Preencher área entre as curvas (força motriz)
ax.fill_between(x, T_agua, T_gases, alpha=0.2, color='orange', label='Força motriz')

# Marcar a transição entre zonas
ax.axvline(x=x_pre, color='gray', linestyle='--', linewidth=1.5)
ax.text(x_pre/2, 450, 'Pré-aquecimento', ha='center', fontsize=11, style='italic')
ax.text((x_pre + 1)/2, 450, 'Vaporização', ha='center', fontsize=11, style='italic')

ax.set_xlabel('Posição no Trocador', fontsize=12)
ax.set_ylabel('Temperatura [°C]', fontsize=12)
ax.set_title('Perfil de Temperatura na Caldeira Aquatubular (Duas Etapas)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(50, 950)

plt.tight_layout()
plt.show()

print(f'\n💡 Interpretação:')
print(f'  A LMTD de pré-aquecimento ({LMTD_pre:.0f}°C) é muito maior que a de')
print(f'  vaporização ({LMTD_vap:.0f}°C), pois nesta última a água está a T constante.')

In [ ]:
# ============================================================
# PASSO 5: ÁREAS REQUERIDAS
# ============================================================

# Área para pré-aquecimento
A_req_pre = Q_pre / (U_o * F_correcao * LMTD_pre)   # [m²]

# Área para vaporização
A_req_vap = Q_vap / (U_o * F_correcao * LMTD_vap)   # [m²]

# Área total requerida
A_req_total = A_req_pre + A_req_vap   # [m²]

print('=' * 60)
print('PASSO 5: ÁREAS REQUERIDAS')
print('=' * 60)
print(f'\n📊 Áreas de troca:')
print(f'  • Pré-aquecimento: {A_req_pre:.2f} m²')
print(f'  • Vaporização: {A_req_vap:.2f} m²')
print(f'  • TOTAL requerida: {A_req_total:.2f} m²')
print(f'  • Disponível (A_o): {A_o:.2f} m²')

if A_req_total <= A_o:
    margem_area = (A_o / A_req_total - 1) * 100
    print(f'\n✅ Projeto ATENDE à demanda (margem de {margem_area:.1f}%)')
else:
    deficit_area = A_req_total - A_o
    print(f'\n❌ Projeto NÃO ATENDE. Déficit de {deficit_area:.2f} m²')
    print(f'  Solução: aumentar N_t ou L_tubo')

---
## 📊 Seção 8: Análise Quantitativa — Validação ε-NTU

In [ ]:
# ============================================================
# PASSO 6: VALIDAÇÃO PELO MÉTODO ε-NTU
# ============================================================

print('=' * 60)
print('PASSO 6: MÉTODO ε-NTU (VALIDAÇÃO CRUZADA)')
print('=' * 60)

# Capacidades térmicas
C_h = m_h * cp_h   # Lado gases [W/K]

# Para a zona de vaporização: C_c → ∞ (mudança de fase)
# Portanto C_r = C_min / C_max = 0
C_min = C_h
C_r = 0.0

# NTU baseado na área disponível e U calculado
NTU = U_o * A_o / C_min

# Efetividade para C_r = 0 (independente da configuração)
epsilon = 1.0 - np.exp(-NTU)

# Calor máximo possível
Q_max = C_min * (T_h_in - T_sat)

# Calor real transferido
Q_real = epsilon * Q_max

# Temperatura de saída real dos gases
T_h_out_NTU = T_h_in - Q_real / C_h

print(f'\n📊 Capacidades Térmicas:')
print(f'  • C_h (gases): {C_h:.2f} W/K')
print(f'  • C_c (vaporização): ∞ (mudança de fase)')
print(f'  • C_r = C_min/C_max: {C_r:.4f}')
print(f'\n🧮 NTU e Efetividade:')
print(f'  • NTU = U_o·A_o/C_min: {NTU:.4f}')
print(f'  • ε = 1 - exp(-NTU): {epsilon:.4f} ({epsilon*100:.2f}%)')
print(f'\n🔍 Desempenho:')
print(f'  • Q_max possível: {Q_max/1e3:.2f} kW')
print(f'  • Q_real transferido: {Q_real/1e3:.2f} kW')
print(f'  • T_h_out real: {T_h_out_NTU:.2f}°C')
print(f'  • Q_req (projeto): {Q_req/1e3:.2f} kW')

if Q_real >= Q_req:
    margem_NTU = (Q_real / Q_req - 1) * 100
    print(f'\n✅ O trocador ATENDE à demanda (margem de {margem_NTU:.1f}%)')
else:
    deficit = (Q_req - Q_real) / 1e3
    NTU_req = -np.log(1.0 - Q_req / Q_max)
    A_req_NTU = NTU_req * C_min / U_o
    print(f'\n⚠️ O trocador NÃO ATENDE. Déficit de {deficit:.2f} kW')
    print(f'  • NTU necessário: {NTU_req:.4f}')
    print(f'  • Área necessária: {A_req_NTU:.2f} m²')

In [ ]:
# ============================================================
# GRÁFICO: COMPARAÇÃO FLAMOTUBULAR vs. AQUATUBULAR
# ============================================================

# Resultados da caldeira flamotubular (EC 9.7)
# h_i_flamotubular = 68.55 W/(m²·K) (gases DENTRO dos tubos)
# h_o_flamotubular = 4000 W/(m²·K) (ebulição em poça)
# U_o_flamotubular = 54.07 W/(m²·K)
# A_req_flamotubular = 47.28 m²

h_i_flamo = 68.55
h_o_flamo = 4000.0
U_o_flamo = 54.07
A_req_flamo = 47.28

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Painel 1: Coeficientes convectivos ---
config = ['Flamotubular', 'Aquatubular']
h_int = [h_i_flamo, h_i_agua]
h_ext = [h_o_flamo, h_o_gases]

x_pos = np.arange(len(config))
width = 0.35

axes[0].bar(x_pos - width/2, h_int, width, label='h interno (tubos)',
            color='#3498db', edgecolor='black', linewidth=0.8)
axes[0].bar(x_pos + width/2, h_ext, width, label='h externo (casco)',
            color='#e74c3c', edgecolor='black', linewidth=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(config)
axes[0].set_ylabel('h [W/(m²·K)]', fontsize=12)
axes[0].set_title('(a) Coeficientes Convectivos', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# --- Painel 2: Coeficiente global U ---
U_vals = [U_o_flamo, U_o]
cores = ['#e67e22', '#27ae60']
bars = axes[1].bar(config, U_vals, color=cores, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, U_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('U_o [W/(m²·K)]', fontsize=12)
axes[1].set_title('(b) Coeficiente Global U', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# --- Painel 3: Área requerida ---
A_vals = [A_req_flamo, A_req_total]
bars = axes[2].bar(config, A_vals, color=cores, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, A_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[2].axhline(A_o, color='red', linestyle='--', linewidth=1.5,
                label=f'Área disponível: {A_o:.1f} m²')
axes[2].set_ylabel('Área [m²]', fontsize=12)
axes[2].set_title('(c) Área Requerida', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Comparação: Caldeira Flamotubular vs. Aquatubular',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABELA COMPARATIVA
# ============================================================

dados_tabela = {
    'Grandeza': [
        'Fluido nos tubos',
        'Fluido no casco',
        'h interno [W/(m²·K)]',
        'h externo [W/(m²·K)]',
        'U_o [W/(m²·K)]',
        'Área requerida [m²]',
        'Área disponível [m²]',
        'Q transferido [kW]',
        'Atende à demanda?'
    ],
    'Flamotubular (EC 9.7)': [
        'Gases (Dittus-Boelter)',
        'Água (pool boiling)',
        f'{h_i_flamo:.1f}',
        f'{h_o_flamo:.0f}',
        f'{U_o_flamo:.1f}',
        f'{A_req_flamo:.1f}',
        f'{A_o:.1f}',
        f'{Q_req/1e3:.1f}',
        '✅ SIM'
    ],
    'Aquatubular (este EC)': [
        'Água (Dittus-Boelter)',
        'Gases (Bell-Delaware)',
        f'{h_i_agua:.1f}',
        f'{h_o_gases:.1f}',
        f'{U_o:.1f}',
        f'{A_req_total:.1f}',
        f'{A_o:.1f}',
        f'{Q_req/1e3:.1f}',
        '✅ SIM' if A_req_total <= A_o else '❌ NÃO'
    ]
}

df_comp = pd.DataFrame(dados_tabela)
display(Markdown('### Tabela Comparativa: Flamotubular vs. Aquatubular'))
display(Markdown(df_comp.to_markdown(index=False)))

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos a sensibilidade do coeficiente global U ao espaçamento entre chicanas e à velocidade dos gases.

In [ ]:
# ============================================================
# SENSIBILIDADE: ESPAÇAMENTO ENTRE CHICANAS
# ============================================================

L_b_range = np.linspace(0.10, 0.50, 20)   # [m]
h_o_range = []
U_o_range = []

for L_b_test in L_b_range:
    # Recalcular área de escoamento
    S_m_test = (p_T - D_o) * L_b_test * D_s / p_T

    # Velocidade e Reynolds dos gases
    u_gases_test = m_h / (rho_h * S_m_test)
    Re_gases_test = rho_h * u_gases_test * D_o / mu_h

    # Nusselt por Zukauskas
    if Re_gases_test > 0:
        Nu_ideal_test = C_zuk * Re_gases_test**m_zuk * Pr_h**0.36
        h_ideal_test = Nu_ideal_test * k_h / D_o
        h_o_test = h_ideal_test * J_total
    else:
        h_o_test = 0.0

    # Coeficiente global
    R_conv_o_test = 1.0 / (h_o_test * A_o) if h_o_test > 0 else 1e10
    R_total_test = R_conv_i + R_fouling_i + R_wall + R_fouling_o + R_conv_o_test
    U_o_test = 1.0 / (R_total_test * A_o)

    h_o_range.append(h_o_test)
    U_o_range.append(U_o_test)

# Gráfico
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(L_b_range * 1000, h_o_range, 'r-o', linewidth=2, markersize=5)
ax1.axvline(L_b * 1000, color='green', linestyle='--', linewidth=1.5,
            label=f'Caso base: L_b = {L_b*1000:.0f} mm')
ax1.set_xlabel('Espaçamento entre chicanas $L_b$ [mm]', fontsize=12)
ax1.set_ylabel('$h_o$ [W/(m²·K)]', fontsize=12)
ax1.set_title('(a) Efeito do Espaçamento em $h_o$', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(L_b_range * 1000, U_o_range, 'b-s', linewidth=2, markersize=5)
ax2.axvline(L_b * 1000, color='green', linestyle='--', linewidth=1.5,
            label=f'Caso base: L_b = {L_b*1000:.0f} mm')
ax2.set_xlabel('Espaçamento entre chicanas $L_b$ [mm]', fontsize=12)
ax2.set_ylabel('$U_o$ [W/(m²·K)]', fontsize=12)
ax2.set_title('(b) Efeito do Espaçamento em $U_o$', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Sensibilidade ao espaçamento das chicanas:')
print(f'  • L_b = 100 mm: h_o = {h_o_range[0]:.1f}, U_o = {U_o_range[0]:.1f} W/(m²·K)')
print(f'  • L_b = 300 mm (base): h_o = {h_o_gases:.1f}, U_o = {U_o:.1f} W/(m²·K)')
print(f'  • L_b = 500 mm: h_o = {h_o_range[-1]:.1f}, U_o = {U_o_range[-1]:.1f} W/(m²·K)')

> 💡 **Observação:**
> - Reduzir o espaçamento entre chicanas aumenta a velocidade dos gases e, consequentemente, $h_o$.
> - Porém, chicanas muito próximas aumentam a queda de pressão e o risco de erosão.
> - O espaçamento ideal equilibra transferência de calor e queda de pressão.

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Configuração física correta:** Na caldeira aquatubular, os gases escoam **transversalmente** ao feixe de tubos com chicanas. O método de Bell-Delaware é o adequado, ao contrário da flamotubular onde os gases estão dentro dos tubos (Dittus-Boelter).

2. **Coeficientes convectivos:** O coeficiente do lado da água ($h_i \approx 2000$–$5000$ W/(m²·K)) é muito maior que o do lado dos gases ($h_o \approx 50$–$200$ W/(m²·K)). A resistência dominante é a convecção dos gases no casco.

3. **Comparação com flamotubular:** A caldeira aquatubular tem coeficiente global tipicamente **menor** que a flamotubular para a mesma capacidade, pois a convecção dos gases no casco é menos eficiente que a convecção interna nos tubos. Porém, a aquatubular permite pressões de operação mais elevadas.

4. **Vantagens da aquatubular:**
   - Maior pressão de operação (tubos de pequeno diâmetro resistem melhor)
   - Maior capacidade de vapor (escalável)
   - Manutenção mais fácil (acesso ao casco)
   - Resposta mais rápida a variações de carga

5. **Desvantagens da aquatubular:**
   - Custo inicial mais elevado
   - Tratamento de água mais rigoroso (tubos de pequeno diâmetro são sensíveis a incrustação)
   - Coeficiente global menor para a mesma geometria

### Limitações do Modelo

- **Regime permanente:** Não considera transientes (partida, variação de carga).
- **Fatores de Bell-Delaware aproximados:** Os fatores $J_c$, $J_l$, $J_b$, $J_s$, $J_w$ foram estimados. Em projetos detalhados, devem ser calculados pelas correlações completas.
- **Sem flow boiling detalhado:** A vaporização dentro dos tubos foi tratada de forma simplificada (LMTD com T constante). Em projetos detalhados, usar correlações de Chen ou Gungor-Winterton.
- **Propriedades constantes:** Não considera variação de propriedades com temperatura.
- **Sem radiação dos gases:** Em caldeiras reais, a radiação dos gases quentes (CO₂, H₂O) é significativa e deve ser incluída.

### Aplicações Práticas

- **Indústria de grande porte:** Geração de vapor em alta pressão para turbinas.
- **Usinas termelétricas:** Caldeiras aquatubulares são padrão em UTEs de médio/grande porte.
- **Indústria química:** Processos que exigem vapor em alta pressão.
- **Cogeração:** Produção combinada de vapor e eletricidade.

### Conexão com a Teoria

> 📖 **Referência:** O método de Bell-Delaware é descrito em Shah & Sekulić (2003), Cap. 2, e Kakaç, Liu & Majumdar (2012), Cap. 4. A correlação de Zukauskas para escoamento cruzado segue Incropera et al. (2017), Cap. 7, Seção 7.6. A comparação entre configurações de caldeiras segue Lugon Jr. et al. (2026), Cap. 9.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 9](../notebooks/09_Trocadores_Calor.ipynb)
- [💡 Caso 9.7 — Caldeira Flamotubular](EC_09_07_Caldeira_Flamotubular.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 9.X2

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>